# SMILES label prediction with a classical baseline
This notebook trains class-weighted logistic regression using 2,048-bit Morgan fingerprints. It evaluates an untouched 20% holdout and five-fold cross-validation on the training partition. The rare positive class makes average precision more informative than accuracy. No label meaning is assumed.

Run cells in order. Dependencies: `pandas numpy scipy scikit-learn rdkit nbformat`. Install with `%pip install pandas numpy scipy scikit-learn rdkit nbformat` if needed. Edit `DATA_DIR` for another machine. The source files are never modified.

In [ ]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn, rdkit
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (average_precision_score, roc_auc_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
DATA_DIR = Path(r"C:\Users\addijk\OneDrive - UvA\Documenten\quantum\dreamQ\phase1")
SEED = 42
print('Python', platform.python_version(), 'RDKit', rdkit.__version__, 'sklearn', sklearn.__version__)


Python 3.12.14 RDKit 2026.09.1 sklearn 1.9.1


In [ ]:
label_path = DATA_DIR / 'retrospective_revealed_labels.csv'
raw = pd.read_csv(label_path)
assert {'CatalogID', 'Label', 'RDKit_SMILES'} <= set(raw.columns)
assert raw.Label.notna().all() and set(raw.Label.unique()) == {0, 1}
print('File SHA256:', hashlib.sha256(label_path.read_bytes()).hexdigest())
print('Raw rows:', len(raw), '\nLabel counts:\n', raw.Label.value_counts().sort_index().to_string())
mols = [Chem.MolFromSmiles(s) if isinstance(s, str) else None for s in raw.RDKit_SMILES]
valid = np.array([m is not None for m in mols])
df = raw.loc[valid].copy()
df['canonical'] = [Chem.MolToSmiles(m, isomericSmiles=True) for m in mols if m is not None]
conflicts = df.groupby('canonical').Label.nunique()
conflicting = set(conflicts[conflicts > 1].index)
conflict_rows = int(df.canonical.isin(conflicting).sum())
df = df.loc[~df.canonical.isin(conflicting)].copy()
duplicates = int(df.canonical.duplicated().sum())
df = df.drop_duplicates('canonical').reset_index(drop=True)
print('Invalid SMILES:', int((~valid).sum()), 'Conflicting-label rows excluded:', conflict_rows,
      'Duplicate structures removed:', duplicates, 'Usable rows:', len(df))
print(df.Label.value_counts().sort_index().to_string())


File SHA256: 55fc2752c75a9aeb1aa0a9b10527768e4ce7f899f82d2efd99b31d41ef9223d5
Raw rows: 6580 
Label counts:
 Label
0    6541
1      39


Invalid SMILES: 0 Conflicting-label rows excluded: 0 Duplicate structures removed: 1 Usable rows: 6579
Label
0    6540
1      39


## Features and evaluation
Morgan radius 2 fingerprints encode local atom environments (including chirality). IDs and labels never enter the feature matrix. Exact canonical duplicates are removed before a stratified 80/20 split (seed 42). Fixed L2 logistic regression uses C=1 and balanced class weights; the decision threshold is fixed at 0.5. There is no tuning against the holdout. Training-only five-fold CV measures variation. This random split can still share related chemical scaffolds, so it does not establish performance on new chemistry.

In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)
def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    return generator.GetFingerprintAsNumPy(mol)
X = np.stack([fingerprint(s) for s in df.canonical])
y = df.Label.to_numpy(dtype=int)
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED)
assert set(df.canonical.iloc[train_idx]).isdisjoint(set(df.canonical.iloc[test_idx]))
X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
print('Train:', len(y_train), 'positives:', int(y_train.sum()))
print('Holdout:', len(y_test), 'positives:', int(y_test.sum()))
def new_model():
    return LogisticRegression(C=1.0, class_weight='balanced', solver='liblinear', max_iter=2000, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_validate(new_model(), X_train, y_train, cv=cv,
    scoring={'AP':'average_precision', 'ROC_AUC':'roc_auc', 'balanced_accuracy':'balanced_accuracy'}, n_jobs=1)
cv_table = pd.DataFrame({k[5:]:v for k,v in cv_scores.items() if k.startswith('test_')})
print('Training CV by fold:\n', cv_table.to_string(index=False))
print('CV mean and sample SD:\n', cv_table.agg(['mean','std']).to_string())


Train: 5263 positives: 31
Holdout: 1316 positives: 8


Training CV by fold:
       AP  ROC_AUC  balanced_accuracy
0.339564 0.973416           0.582378
0.660444 0.854982           0.748567
0.719202 0.992762           0.927137
0.555064 0.863448           0.832377
0.223789 0.860102           0.580943
CV mean and sample SD:
             AP   ROC_AUC  balanced_accuracy
mean  0.499613  0.908942           0.734281
std   0.211457  0.068098           0.152977


In [ ]:
model = new_model().fit(X_train, y_train)
dummy = DummyClassifier(strategy='prior').fit(X_train, y_train)
def evaluate(estimator):
    p = estimator.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    return {'Average precision':average_precision_score(y_test,p), 'ROC AUC':roc_auc_score(y_test,p),
        'Accuracy':accuracy_score(y_test,pred), 'Balanced accuracy':balanced_accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred,zero_division=0), 'Recall':recall_score(y_test,pred),
        'F1':f1_score(y_test,pred,zero_division=0)}, p, pred
metrics, probabilities, predictions = evaluate(model)
baseline_metrics, _, _ = evaluate(dummy)
results = pd.DataFrame({'Logistic regression':metrics, 'Prior baseline':baseline_metrics})
cm = confusion_matrix(y_test, predictions, labels=[0,1])
print(results.to_string(float_format=lambda v:f'{v:.4f}'))
print('Confusion matrix (rows true 0/1; columns predicted 0/1):\n', cm)
print('Holdout positive prevalence:', y_test.mean())
# Holdout predictions remain from the training-only model, before the final refit.
holdout_predictions = df.iloc[test_idx][['CatalogID','RDKit_SMILES','Label']].copy()
holdout_predictions['score'] = probabilities
holdout_predictions['predicted_label'] = predictions
holdout_predictions.sort_values('score', ascending=False).head(10)


                   Logistic regression  Prior baseline
Average precision               0.2676          0.0061
ROC AUC                         0.8831          0.5000
Accuracy                        0.9924          0.9939
Balanced accuracy               0.6235          0.5000
Precision                       0.3333          0.0000
Recall                          0.2500          0.0000
F1                              0.2857          0.0000
Confusion matrix (rows true 0/1; columns predicted 0/1):
 [[1304    4]
 [   6    2]]
Holdout positive prevalence: 0.0060790273556231


,CatalogID,RDKit_SMILES,Label,score,predicted_label
15,Z359457804,O=C(Nc1cccc(-c2ccn[nH]2)c1)c1ccc2ccccc2n1,1,0.946267,1
3682,Z2125630906,COc1c(C(=O)Nc2[nH]nc(C)c2C#N)cnc2c1cnn2C,0,0.903056,1
35,Z1445518982,O=C(O)CCc1ccc(NC(=O)c2cccc3cn[nH]c23)cc1,1,0.864735,1
1335,Z442962706,O=C(Nc1cccc(C#Cc2ccccn2)c1)c1ccc(-c2ccccc2)[nH...,0,0.632243,1
619,Z788118638,N#Cc1c(NC(=O)Nc2cccnc2)sc2c1CCNC2,0,0.610285,1
2699,Z27033169,O=C(Nc1nc2c(F)c(F)c(F)cc2s1)c1ccc(Cl)cc1[N+](=...,0,0.586646,1
5007,Z1603682130,O=C(O)CC(NC(=O)c1cccc2cn[nH]c12)c1ccc(Cl)c(F)c1,0,0.451474,0
3676,Z2129063473,O=C(Nc1cccnc1)c1ccnc(-c2ccc(Cl)cc2)n1,0,0.432816,0
3014,Z240721912,C#Cc1cccc(NC(=O)c2ccc(=O)[nH]n2)c1,0,0.428337,0
4008,Z19739821,N#Cc1ccc(OCC(=O)Nc2nc3ccc(F)cc3s2)cc1,0,0.384625,0
